# Task 4: Portfolio Optimization
## Efficient Frontier and Optimal Portfolio Construction

This notebook implements portfolio optimization using:
- Expected returns from TSLA forecast (Task 3)
- Historical returns for BND and SPY
- Covariance matrix computation
- Efficient Frontier generation
- Maximum Sharpe Ratio and Minimum Volatility portfolios

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("Libraries imported successfully!")

## 1. Load Data and Prepare Expected Returns

In [ ]:
# Load historical price data
prices = pd.read_csv('../data/raw/stock_prices.csv', index_col=0, parse_dates=True)
returns = pd.read_csv('../data/processed/daily_returns.csv', index_col=0, parse_dates=True)

# Load TSLA forecast from Task 3
try:
    tsla_forecast = pd.read_csv('../data/processed/tsla_forecast.csv')
    use_forecast = True
    print("TSLA forecast loaded from Task 3")
except:
    use_forecast = False
    print("Note: TSLA forecast not found. Using historical average.")

print(f"\nPrice data shape: {prices.shape}")
print(f"Returns data shape: {returns.shape}")

## 2. Expected Returns Preparation

- TSLA: Use forecast expected return from Task 3
- BND and SPY: Use historical average annualized returns

In [ ]:
# Calculate expected returns
TICKERS = ['TSLA', 'BND', 'SPY']
TRADING_DAYS = 252
RISK_FREE_RATE = 0.02  # 2% annual risk-free rate

# Historical average annualized returns
historical_returns = returns.mean() * TRADING_DAYS

print("=== Historical Annualized Returns ===")
print(historical_returns.round(4))

# Prepare expected returns vector
if use_forecast:
    # Use TSLA forecast return, historical for BND and SPY
    expected_returns = pd.Series({
        'TSLA': tsla_forecast['Annualized_Return'].iloc[0],
        'BND': historical_returns['BND'],
        'SPY': historical_returns['SPY']
    })
    print(f"\n=== Using TSLA Forecast Expected Return ===")
    print(f"TSLA (forecast): {expected_returns['TSLA']:.2%}")
else:
    expected_returns = historical_returns

print(f"\n=== Expected Returns for Portfolio Optimization ===")
for ticker in TICKERS:
    print(f"{ticker}: {expected_returns[ticker]:.2%}")

## 3. Covariance Matrix Computation

In [ ]:
# Calculate covariance matrix from daily returns
cov_matrix = returns[TICKERS].cov() * TRADING_DAYS  # Annualized

print("=== Annualized Covariance Matrix ===")
display(cov_matrix.round(6))

# Volatilities (from diagonal)
volatilities = np.sqrt(np.diag(cov_matrix))
print(f"\n=== Annualized Volatilities ===")
for i, ticker in enumerate(TICKERS):
    print(f"{ticker}: {volatilities[i]:.2%}")

In [ ]:
# Covariance Matrix Heatmap Visualization
fig, ax = plt.subplots(figsize=(10, 8))

# Create heatmap
sns.heatmap(cov_matrix, annot=True, fmt='.6f', cmap='RdYlGn_r', 
            xticklabels=TICKERS, yticklabels=TICKERS, ax=ax,
            annot_kws={'size': 12})

ax.set_title('Annualized Covariance Matrix Heatmap', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../data/processed/covariance_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved to data/processed/covariance_heatmap.png")

In [ ]:
# Correlation matrix (normalized view)
corr_matrix = returns[TICKERS].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.4f', cmap='coolwarm', 
            xticklabels=TICKERS, yticklabels=TICKERS, ax=ax,
            annot_kws={'size': 12}, vmin=-1, vmax=1)

ax.set_title('Correlation Matrix Heatmap', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../data/processed/correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved to data/processed/correlation_heatmap.png")

## 4. Efficient Frontier Generation

Using scipy.optimize to minimize volatility for various target returns.

In [ ]:
def portfolio_return(weights, expected_returns):
    """Calculate portfolio expected return."""
    return np.dot(weights, expected_returns)

def portfolio_volatility(weights, cov_matrix):
    """Calculate portfolio volatility."""
    return np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))

def portfolio_sharpe(weights, expected_returns, cov_matrix, rf=RISK_FREE_RATE):
    """Calculate portfolio Sharpe ratio."""
    ret = portfolio_return(weights, expected_returns)
    vol = portfolio_volatility(weights, cov_matrix)
    return (ret - rf) / vol

def minimize_volatility(target_return, expected_returns, cov_matrix):
    """Find minimum volatility portfolio for target return."""
    n_assets = len(expected_returns)
    
    # Constraints
    constraints = [
        {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},  # Weights sum to 1
        {'type': 'eq', 'fun': lambda w: portfolio_return(w, expected_returns) - target_return}  # Target return
    ]
    
    # Bounds: weights between 0 and 1 (no short selling)
    bounds = tuple((0, 1) for _ in range(n_assets))
    
    # Initial guess: equal weights
    init_weights = np.ones(n_assets) / n_assets
    
    result = minimize(
        lambda w: portfolio_volatility(w, cov_matrix),
        init_weights,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints
    )
    
    if result.success:
        return result.x
    return None

print("Optimization functions defined.")

In [ ]:
# Generate efficient frontier
n_points = 100
min_ret = expected_returns.min()
max_ret = expected_returns.max()
target_returns = np.linspace(min_ret, max_ret, n_points)

frontier_volatilities = []
frontier_returns = []
frontier_weights = []

print(f"Generating efficient frontier with {n_points} points...")

for target_ret in target_returns:
    weights = minimize_volatility(target_ret, expected_returns.values, cov_matrix.values)
    if weights is not None:
        vol = portfolio_volatility(weights, cov_matrix.values)
        frontier_volatilities.append(vol)
        frontier_returns.append(target_ret)
        frontier_weights.append(weights)

print(f"Generated {len(frontier_volatilities)} frontier points.")

## 5. Maximum Sharpe Ratio Portfolio

In [ ]:
# Optimize for maximum Sharpe ratio
n_assets = len(TICKERS)

constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
bounds = tuple((0, 1) for _ in range(n_assets))
init_weights = np.ones(n_assets) / n_assets

# Minimize negative Sharpe (equivalent to maximizing Sharpe)
result = minimize(
    lambda w: -portfolio_sharpe(w, expected_returns.values, cov_matrix.values),
    init_weights,
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)

max_sharpe_weights = result.x
max_sharpe_return = portfolio_return(max_sharpe_weights, expected_returns.values)
max_sharpe_vol = portfolio_volatility(max_sharpe_weights, cov_matrix.values)
max_sharpe_ratio = portfolio_sharpe(max_sharpe_weights, expected_returns.values, cov_matrix.values)

print("="*60)
print("MAXIMUM SHARPE RATIO PORTFOLIO")
print("="*60)
print(f"\nPortfolio Weights:")
for i, ticker in enumerate(TICKERS):
    print(f"  {ticker}: {max_sharpe_weights[i]:.2%}")
print(f"\nExpected Annual Return: {max_sharpe_return:.2%}")
print(f"Expected Annual Volatility: {max_sharpe_vol:.2%}")
print(f"Sharpe Ratio: {max_sharpe_ratio:.4f}")

## 6. Minimum Volatility Portfolio

In [ ]:
# Optimize for minimum volatility
result = minimize(
    lambda w: portfolio_volatility(w, cov_matrix.values),
    init_weights,
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)

min_vol_weights = result.x
min_vol_return = portfolio_return(min_vol_weights, expected_returns.values)
min_vol_vol = portfolio_volatility(min_vol_weights, cov_matrix.values)
min_vol_sharpe = portfolio_sharpe(min_vol_weights, expected_returns.values, cov_matrix.values)

print("="*60)
print("MINIMUM VOLATILITY PORTFOLIO")
print("="*60)
print(f"\nPortfolio Weights:")
for i, ticker in enumerate(TICKERS):
    print(f"  {ticker}: {min_vol_weights[i]:.2%}")
print(f"\nExpected Annual Return: {min_vol_return:.2%}")
print(f"Expected Annual Volatility: {min_vol_vol:.2%}")
print(f"Sharpe Ratio: {min_vol_sharpe:.4f}")

## 7. Efficient Frontier Visualization

In [ ]:
# Create efficient frontier plot
fig, ax = plt.subplots(figsize=(14, 10))

# Plot efficient frontier
ax.plot(frontier_volatilities, frontier_returns, 'b-', linewidth=3, 
        label='Efficient Frontier')

# Plot individual assets
colors = {'TSLA': '#E31937', 'BND': '#003366', 'SPY': '#FF6600'}
for ticker in TICKERS:
    ax.scatter(volatilities[TICKERS.index(ticker)], expected_returns[ticker],
               s=200, c=colors[ticker], marker='o', label=ticker, 
               edgecolors='black', linewidths=2, zorder=5)

# Plot Maximum Sharpe Portfolio
ax.scatter(max_sharpe_vol, max_sharpe_return, s=300, c='green', marker='*', 
           label='Max Sharpe Ratio', edgecolors='black', linewidths=2, zorder=6)

# Plot Minimum Volatility Portfolio
ax.scatter(min_vol_vol, min_vol_return, s=300, c='red', marker='*', 
           label='Min Volatility', edgecolors='black', linewidths=2, zorder=6)

# Capital Allocation Line
cal_vol = np.linspace(0, max(frontier_volatilities) * 1.5, 100)
cal_return = RISK_FREE_RATE + max_sharpe_ratio * cal_vol
ax.plot(cal_vol, cal_return, 'g--', linewidth=2, label='Capital Allocation Line')

ax.set_xlabel('Expected Volatility (Standard Deviation)', fontsize=12)
ax.set_ylabel('Expected Return', fontsize=12)
ax.set_title('Efficient Frontier with Optimal Portfolios', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)

# Format axes as percentages
from matplotlib.ticker import FuncFormatter
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:.0%}'))

plt.tight_layout()
plt.savefig('../data/processed/efficient_frontier.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved to data/processed/efficient_frontier.png")

## 8. Portfolio Recommendations Summary

In [ ]:
# Create summary table
print("\n" + "="*80)
print("PORTFOLIO OPTIMIZATION SUMMARY")
print("="*80)

summary_data = []

# Maximum Sharpe Portfolio
summary_data.append({
    'Portfolio': 'Maximum Sharpe Ratio',
    'TSLA Weight': f"{max_sharpe_weights[0]:.2%}",
    'BND Weight': f"{max_sharpe_weights[1]:.2%}",
    'SPY Weight': f"{max_sharpe_weights[2]:.2%}",
    'Expected Return': f"{max_sharpe_return:.2%}",
    'Volatility': f"{max_sharpe_vol:.2%}",
    'Sharpe Ratio': f"{max_sharpe_ratio:.4f}"
})

# Minimum Volatility Portfolio
summary_data.append({
    'Portfolio': 'Minimum Volatility',
    'TSLA Weight': f"{min_vol_weights[0]:.2%}",
    'BND Weight': f"{min_vol_weights[1]:.2%}",
    'SPY Weight': f"{min_vol_weights[2]:.2%}",
    'Expected Return': f"{min_vol_return:.2%}",
    'Volatility': f"{min_vol_vol:.2%}",
    'Sharpe Ratio': f"{min_vol_sharpe:.4f}"
})

# Equal Weight Portfolio (baseline)
equal_weights = np.ones(n_assets) / n_assets
equal_return = portfolio_return(equal_weights, expected_returns.values)
equal_vol = portfolio_volatility(equal_weights, cov_matrix.values)
equal_sharpe = portfolio_sharpe(equal_weights, expected_returns.values, cov_matrix.values)

summary_data.append({
    'Portfolio': 'Equal Weight (Baseline)',
    'TSLA Weight': f"{equal_weights[0]:.2%}",
    'BND Weight': f"{equal_weights[1]:.2%}",
    'SPY Weight': f"{equal_weights[2]:.2%}",
    'Expected Return': f"{equal_return:.2%}",
    'Volatility': f"{equal_vol:.2%}",
    'Sharpe Ratio': f"{equal_sharpe:.4f}"
})

summary_df = pd.DataFrame(summary_data)
display(summary_df)

# Save results
summary_df.to_csv('../data/processed/portfolio_optimization_results.csv', index=False)
print("\nResults saved to data/processed/portfolio_optimization_results.csv")

In [ ]:
# Save optimal weights for backtesting (Task 5)
optimal_weights = pd.DataFrame({
    'Ticker': TICKERS,
    'Max_Sharpe_Weight': max_sharpe_weights,
    'Min_Vol_Weight': min_vol_weights
})

optimal_weights.to_csv('../data/processed/optimal_weights.csv', index=False)

print("\nOptimal weights saved for Task 5 backtesting.")
display(optimal_weights)

## Summary

### Outputs:
1. **Expected Returns**: TSLA forecast + BND/SPY historical
2. **Covariance Matrix**: Annualized from daily returns with heatmap
3. **Efficient Frontier**: 100 points showing risk-return tradeoff
4. **Maximum Sharpe Portfolio**: Highest risk-adjusted return
5. **Minimum Volatility Portfolio**: Lowest risk

### Recommended Portfolio:
Based on the optimization, the **Maximum Sharpe Ratio Portfolio** is recommended for investors seeking optimal risk-adjusted returns.

### Next Steps:
- Task 5: Backtest these portfolios against benchmark strategies